## Initial data processing for quality and leakage prevention

In [ ]:
import pandas as pd
import json
from pathlib import Path
import re

# Carregar o ficheiro JSON

# Do CSM
csm_ic = Path("./../../data/raw_data/csm_incumprimento_contratos_20251029_114925.json")
csm_dv = Path("./../../data/raw_data/csm_violencia_domestica_20251029_131345.json")

# Do DGSI
# Violencia Doméstica
dgsi_dv_trp = Path(
    "./../../data/raw_data/dgsi_violencia_domestica_trp_20251013_175933.json"
)
dgsi_dv_trl = Path(
    "./../../data/raw_data/dgsi_violencia_domestica_trl_20251013_173521.json"
)
dgsi_dv_trc = Path(
    "./../../data/raw_data/dgsi_violencia_domestica_trc_20251014_104127.json"
)
dgsi_dv_tre = Path(
    "./../../data/raw_data/dgsi_violencia_domestica_tre_20251013_171439.json"
)
dgsi_dv_trg = Path(
    "./../../data/raw_data/dgsi_violencia_domestica_trg_20251014_104936.json"
)
dgsi_dv_stj = Path(
    "./../../data/raw_data/dgsi_violencia_domestica_stj_20251016_093545.json"
)

# Incumprimento de Contratos
dgsi_ic_jp = Path(
    "./../../data/raw_data/dgsi_incumprimento_contratos_20251028_130853.json"
)
dgsi_ic_trl = Path(
    "./../../data/raw_data/dgsi_incumprimento_contratos_trl_20251028_152622.json"
)
dgsi_ic_trp = Path(
    "./../../data/raw_data/dgsi_incumprimento_contratos_trp_20251028_162602.json"
)
dgsi_ic_trc = Path(
    "./../../data/raw_data/dgsi_incumprimento_contratos_trc_20251028_154117.json"
)
dgsi_ic_tre = Path(
    "./../../data/raw_data/dgsi_incumprimento_contratos_tre_20251028_141118.json"
)
dgsi_ic_trg = Path(
    "./../../data/raw_data/dgsi_incumprimento_contratos_trg_20251028_171624.json"
)
dgsi_ic_stj = Path(
    "./../../data/raw_data/dgsi_incumprimento_contratos_stj_20251028_135712.json"
)

ic_cases_paths = [
    csm_ic,
    dgsi_ic_jp,
    dgsi_ic_trl,
    dgsi_ic_trp,
    dgsi_ic_trc,
    dgsi_ic_tre,
    dgsi_ic_trg,
    dgsi_ic_stj,
]

dv_cases_paths = [
    csm_dv,
    dgsi_dv_trp,
    dgsi_dv_trl,
    dgsi_dv_trc,
    dgsi_dv_tre,
    dgsi_dv_trg,
    dgsi_dv_stj,
]


# Verificar se o ficheiro existe
def create_dataframe_from_paths(paths):
    dataframes = []
    for i, path in enumerate(paths):
        if not path.exists():
            print(f"Error: File not found at {path.absolute()}")
            print(f"Current working directory: {Path.cwd()}")
        else:
            with open(path, encoding="utf-8") as f:
                data = json.load(f)

            acordaos = pd.json_normalize(data, sep="_")
            dataframes.append(acordaos)

    return dataframes

In [ ]:
dataframes_dv = create_dataframe_from_paths(dv_cases_paths)

In [ ]:
dataframes_ic = create_dataframe_from_paths(ic_cases_paths)

In [ ]:
def concatenate_case_dataframes():
    """
    Concatena todos os dataframes de cada tipo de caso em 2 dataframes finais:
    - df_dv_all: Todos os casos de Violência Doméstica
    - df_ic_all: Todos os casos de Incumprimento de Contratos

    Returns:
        tuple: (df_dv_all, df_ic_all)
    """
    # Violência Doméstica - concatenar todos os tribunais
    df_dv_all = pd.concat(
        [
            dataframes_dv[0],  # CSM
            dataframes_dv[1],  # TRP
            dataframes_dv[2],  # TRL
            dataframes_dv[3],  # TRC
            dataframes_dv[4],  # TRE
            dataframes_dv[5],  # TRG
            dataframes_dv[6],  # STJ
        ],
        ignore_index=True,
    )

    print(f" Violência Doméstica: {len(df_dv_all)} casos totais")
    print(f"   - CSM: {len(dataframes_dv[0])} casos")
    print(f"   - TRP: {len(dataframes_dv[1])} casos")
    print(f"   - TRL: {len(dataframes_dv[2])} casos")
    print(f"   - TRC: {len(dataframes_dv[3])} casos")
    print(f"   - TRE: {len(dataframes_dv[4])} casos")
    print(f"   - TRG: {len(dataframes_dv[5])} casos")
    print(f"   - STJ: {len(dataframes_dv[6])} casos\n")

    # Incumprimento de Contratos - concatenar todos os tribunais
    df_ic_all = pd.concat(
        [
            dataframes_ic[0],  # CSM
            dataframes_ic[1],  # JP
            dataframes_ic[2],  # TRL
            dataframes_ic[3],  # TRP
            dataframes_ic[4],  # TRC
            dataframes_ic[5],  # TRE
            dataframes_ic[6],  # TRG
            dataframes_ic[7],  # STJ
        ],
        ignore_index=True,
    )

    print(f" Incumprimento de Contratos: {len(df_ic_all)} casos totais")
    print(f"   - CSM: {len(dataframes_ic[0])} casos")
    print(f"   - JP: {len(dataframes_ic[1])} casos")
    print(f"   - TRL: {len(dataframes_ic[2])} casos")
    print(f"   - TRP: {len(dataframes_ic[3])} casos")
    print(f"   - TRC: {len(dataframes_ic[4])} casos")
    print(f"   - TRE: {len(dataframes_ic[5])} casos")
    print(f"   - TRG: {len(dataframes_ic[6])} casos")
    print(f"   - STJ: {len(dataframes_ic[7])} casos\n")

    print(f"Total cases combined: {len(df_dv_all) + len(df_ic_all)}")

    return df_dv_all, df_ic_all


# Criar os 2 dataframes consolidados
df_dv_all, df_ic_all = concatenate_case_dataframes()

# Escolher qual tipo de caso processar
type_case = "ic"  # 'dv' para Violência Doméstica, 'ic' para Incumprimento de Contratos

if type_case == "dv":
    df = df_dv_all
    print(f"🔍 Processando: Violência Doméstica ({len(df)} casos)")
elif type_case == "ic":
    df = df_ic_all
    print(f"🔍 Processando: Incumprimento de Contratos ({len(df)} casos)")
else:
    raise ValueError(f"Tipo de caso inválido: {type_case}. Use 'dv' ou 'ic'.")

## Preparação Inicial dos df's para CLassificação e EDA


In [ ]:
df

In [ ]:
def check_proportion_and_filter(df):
    prop_texto_int_disp = df["texto_integral_disponivel"].isnull().sum() / len(
        df
    )  # Proporção de valores nulos na coluna 'texto_integral_disponivel'
    print(
        f"Proporção de valores nulos em texto_integral_disponivel: {prop_texto_int_disp:.2%}"
    )

    prop_texto_int_disp_falso = (df["texto_integral_disponivel"] == "N").sum() / len(
        df
    )  # Proporção de valores "N" na coluna 'texto_integral_disponivel'
    print(
        f"Proporção de valores N em texto_integral_disponivel: {prop_texto_int_disp_falso:.2%}"
    )

    # prop_texto_int = df['texto_integral_completo'].isnull().sum() / len(df) # Proporção de valores nulos na coluna 'texto_integral_disponivel'
    # print(f'Proporção de valores nulos em texto_integral_completo: {prop_texto_int:.2%}')

    prop_decisao_extraida = df[
        "decisao_extraida_do_texto_integral"
    ].isnull().sum() / len(
        df
    )  # Proporção de valores nulos na coluna 'decisao_extraida_texto_integral'
    print(
        f"Proporção de valores nulos em decisao_extraida_do_texto: {prop_decisao_extraida:.2%}"
    )

    # Create a new dataframe with cases where full text is available (NOT 'N') but decision not extracted
    mask_text_available = df["texto_integral_disponivel"] != "N"

    df_text_available_no_decision = df[
        mask_text_available & df["decisao_extraida_do_texto_integral"].isnull()
    ].copy()

    prop_text_available_no_decision = len(df_text_available_no_decision) / len(df)
    print(
        f'Proporção de casos com texto integral disponível (não "N") mas sem decisão extraída: {prop_text_available_no_decision:.2%}'
    )
    print(
        f'Numero de casos com texto integral disponível (não "N") mas sem decisão extraída: {len(df_text_available_no_decision)}'
    )

    return df_text_available_no_decision

In [ ]:
# Excluir JP apenas para aplicar o drop; guardar os JP à parte
df_jp = df[df["tribunal"].str.startswith("JP_", na=False)].copy()
df_non_jp = df[
    ~df["tribunal"].str.startswith("JP_", na=False)
].copy()  # ~df means ignoring the condition , getting all values different than the condition

# Drop rows com texto_integral_disponivel ausente apenas nos não-JP
df_non_jp = df_non_jp.dropna(subset=["texto_integral_disponivel"], how="all")

# Reunir novamente: manter todos os JP intactos + non-JP filtrados
df = pd.concat([df_non_jp, df_jp], axis=0)

# Opcional: restaurar a ordem original dos índices
df = df.sort_index()

df

In [ ]:
# Capturar o dataframe retornado pela função
df_casos_problematicos = check_proportion_and_filter(df)

print(
    f"Temos {len(df_casos_problematicos)} casos problemáticos onde o texto integral está disponível mas a decisão não foi extraída."
)

In [ ]:
df_casos_problematicos

In [ ]:
# Drop rows with missing data in column: 'decisao_extraida_do_texto_integral'
# df_casos_fixed = df_casos_fixed.dropna(subset=['decisao_extraida_do_texto_integral'], how='all')

df_casos_fixed = df.dropna(subset=["decisao_extraida_do_texto_integral"], how="all")

df_casos_fixed

In [ ]:
df_fixed = df.copy()

# Index both dataframes by 'url' for proper alignment

df_fixed.set_index("url", inplace=True)
df_casos_fixed.set_index("url", inplace=True)

# Update df with values from df_casos
df_fixed.update(df_casos_fixed)

# Reset index if needed
df_fixed.reset_index(inplace=True)
df_casos_fixed.reset_index(inplace=True)

In [ ]:
# drop rows where the column equals "N"
df_fixed = df_fixed[df_fixed["texto_integral_disponivel"] != "N"]


df_fixed

In [ ]:
# Drop rows with missing data in column: 'decisao_extraida_do_texto_integral'
df_fixed = df_fixed.dropna(subset=["decisao_extraida_do_texto_integral"], how="all")

# Drop duplicate rows based on 'n_processo' column, keeping distinct cases
# URL is not enought to identify distinct cases as some cases have multiple URLs due to scraping being done on different databases
df_fixed = df_fixed.drop_duplicates(subset=["n_processo"])

df_fixed

In [ ]:
df_fixed

In [ ]:
def extract_decision_summary(verbose_decision):
    if not verbose_decision:
        return None

    # Convert to lowercase for matching
    text = verbose_decision.lower()
    # Remove punctuation for better matching
    text_clean = re.sub(r"[^\w\s]", " ", text)

    # PRIORITY 1: Check for PARTIAL patterns first
    partial_patterns = [
        r"\bparcial",  # Partial
        r"\bparcialmente",  # Partially
        r"\bem\s+parte",  # In part
        r"\brevogad[oa]\s+parcial",  # Partially revoked
        r"\bprocedente\s+parcial",  # Partially procedent
        r"\bprovido\s+parcial",  # Partially granted
        r"\bconcedid[oa]\s+parcial",  # Partially granted
    ]

    # If ANY partial pattern found, return PARCIAL immediately
    for pattern in partial_patterns:
        if re.search(pattern, text_clean):
            return "PARCIALMENTE PROCEDENTE"

    # PRIORITY 2: Check for FAVORABLE patterns (complete victory - explicitly exclude partials)
    favorable_patterns = [
        # Revocation patterns (complete revocation only - NOT partial)
        r"\brevogad[oa](?!\s*parcial)",  # Revoked but NOT partial
        r"\brevoga[rd](?!\s*parcial)",  # Revoke but NOT partial
        # Procedence patterns (complete only - NOT partial)
        r"\bprocedente(?!\s*(?:parcial|em\s*parte))",  # Procedent but NOT partial
        r"\btotalmente\s+procedente",  # Totally procedent
        # Appeal granted completely (NOT partial)
        r"\bprovido(?!\s*parcial)",  # Granted but NOT partial
        r"\bprovimento(?!\s*parcial)",  # Grant but NOT partial
        # Review granted (NOT partial)
        r"\bconcedid[oa](?!\s*parcial)",  # Granted but NOT partial
        # Decision altered
        r"\balterad[oa]",  # Decision altered
        r"\balterar",  # Alter verb forms
        # Conviction (in civil cases)
        r"\bcondenad[oa]",  # Convicted
    ]

    # Try favorable patterns
    for pattern in favorable_patterns:
        if re.search(pattern, text_clean):
            return "REVOGADA"

    # PRIORITY 3: Check for UNFAVORABLE patterns (explicitly exclude partials)
    unfavorable_patterns = [
        # Any rejection/denial (NOT partial)
        r"\bnegad[oa](?!\s*parcial)",  # Denied but NOT partial
        r"\bnega[rd](?!\s*parcial)",  # Deny but NOT partial
        # Any confirmation (maintains previous decision - NOT partial)
        r"\bconfirmad[oa](?!\s*parcial)",  # Confirmed but NOT partial
        r"\bconfirma[rd](?!\s*parcial)",  # Confirm but NOT partial
        r"\bconfirma[çc][aã]o(?!\s*parcial)",  # Confirmation but NOT partial
        r"\bmantid[oa](?!\s*parcial)",  # Maintained but NOT partial
        # Action/appeal denied (NOT partial)
        r"\bimprocedente(?!\s*(?:parcial|em\s*parte))",  # Improcedent but NOT partial
        r"\bimprocedência",  # Improcedence
        # Specific appeal outcomes
        r"\bapela[cç][aã]o\s+improcedente",  # Appeal improcedent
        # Appeal denials (NOT partial)
        r"\bnegado\s+provimento(?!\s*parcial)",  # Appeal denied but NOT partial
        r"\brecurso.*improcedente",  # Appeal unsuccessful
    ]

    # Try unfavorable patterns
    for pattern in unfavorable_patterns:
        if re.search(pattern, text_clean):
            return "NEGADO PROVIMENTO"

    return None

In [ ]:
df_class_fixed = df_fixed.copy()

# Filter that selects rows where 'decisao' is null or empty
mask = df_class_fixed["decisao"].isnull() | df_class_fixed["decisao"].astype(
    str
).str.strip().eq("")

for index in df_class_fixed[mask].index:
    verbose_decision = df_class_fixed.at[index, "decisao_extraida_do_texto_integral"]
    summary_decision = extract_decision_summary(verbose_decision)
    # assign into decisao (and optionally keep decisao_resumo_extraida)
    df_class_fixed.at[index, "decisao"] = summary_decision
    df_class_fixed.at[index, "decisao_resumo_extraida"] = summary_decision
    print(index, summary_decision)

In [ ]:
df_class_fixed

In [ ]:
# Drop columns: 'url', 'tribunal' and 18 other columns
df_class = df_class_fixed.drop(
    columns=[
        "url",
        "tribunal",
        "tipo_direito",
        "tipo_caso",
        "n_processo",
        "juiz_relator",
        "data_acordao",
        "sumario",
        "votacao",
        "meio_processual",
        "texto_integral_disponivel",
        "texto_integral_completo",
        "decisao_extraida_do_texto_integral",
        "metadata_decisao_extraction_method",
        "metadata_decisao_confidence",
        "metadata_decisao_keyword_found",
        "metadata_decisao_requires_manual_review",
        "metadata_decisao_review_reason",
        "metadata_decisao_document_position",
        "decisao_resumo_extraida",
    ]
)

df_eda = df_class_fixed.drop(
    columns=[
        "tipo_direito",
        "tipo_caso",
        "sumario",
        "votacao",
        "meio_processual",
        "texto_integral_disponivel",
        "texto_integral_completo",
        "decisao_extraida_do_texto_integral",
        "metadata_decisao_extraction_method",
        "metadata_decisao_confidence",
        "metadata_decisao_keyword_found",
        "metadata_decisao_requires_manual_review",
        "metadata_decisao_review_reason",
        "metadata_decisao_document_position",
        "decisao_resumo_extraida",
    ]
)


# Deixar tbm descritores,full_text,url,n_processo (para EDA)

In [ ]:
df_class

In [ ]:
df_eda

In [ ]:
"""This function had a bug, where it misclassified some partial decisions
as favorable since it prioritized favorable patterns search, which in turn created mismatches with the ternary dataset.
The solution is to also search for partial terms first, and when found, classify them as unfavorable.
Then we can normally proceed with finding favorable and unfavorable patterns.
"""

# def extract_decision_binary_from_summary(summary_decision):
#     """

#     MELHOR ATE AGORA


#     Classifica decisões sumárias em binário com lógica "tudo ou nada":
#     - FAVORÁVEL: Decisão completamente positiva para o recorrente
#     - DESFAVORÁVEL: Qualquer coisa que não seja vitória completa

#     REGRA: Parciais = DESFAVORÁVEL (recorrente não obteve tudo)
#     """
#     if not summary_decision or pd.isna(summary_decision):
#         return None

#     # Convert to lowercase and clean
#     text = str(summary_decision).lower().strip()
#     # Remove punctuation for better matching
#     text_clean = re.sub(r'[^\w\s]', ' ', text)

#     # Define STRICTLY FAVORABLE patterns (complete victory only)
#     strictly_favorable_patterns = [
#         # Revocation patterns (any form of complete revocation)
#         r'\brevogad[oa](?!\s*parcial)',     # Revoked but NOT partial
#         r'\brevoga[rd](?!\s*parcial)',      # Revoke but NOT partial

#         # Procedence patterns (complete only)
#         r'\bprocedente(?!\s*(?:parcial|em\s*parte))',  # Procedent but NOT partial

#         # Appeal granted patterns
#         r'\bprovido(?!\s*parcial)',         # Granted but NOT partial
#         r'\bprovimento(?!\s*parcial)',      # Grant but NOT partial

#         # Review/revista granted patterns
#         r'\bconcedid[oa](?!\s*parcial)',    # Granted but NOT partial

#         # Decision altered patterns
#         r'\balterad[oa]',                   # Decision altered
#         r'\balterar',                  # Alter verb forms

#         # Conviction patterns (in civil cases, usually favorable)
#         r'\bcondenad[oa]',                  # Convicted


#     ]

#     # Everything else is UNFAVORABLE (including partials and all rejections)
#     unfavorable_patterns = [
#         # Any rejection/denial
#         r'\bnegad[oa]',                     # Any form of denied
#         r'\bnega[rd]',                      # Deny verb forms

#         # Any confirmation (maintains previous unfavorable decision)
#         r'\bconfirmad[oa]',                 # Any form of confirmed
#         r'\bconfirma[rd]',                  # Confirm verb forms
#         r'\bconfirma[çc][aã]o',            # Confirmation noun
#         r'\bmantid[oa]',                    # Maintained

#         # Action/appeal denied
#         r'\bimprocedente(?!\s*(?:a\s*)?apela[cç][aã]o)', # Improcedent but NOT "appeal improcedent"
#         r'\bimprocedência',                 # Improcedence

#          # Specific appeal outcomes that indicate action lost
#         r'\bapela[cç][aã]o\s+improcedente', # Appeal improcedent
#         r'\bimprocedente\s+a\s+apela[cç][aã]o', # Appeal improcedent

#         # Any partial outcome (treated as unfavorable)
#         r'\bparcial',                       # Any partial
#         r'\bparcialmente',                  # Partially
#         r'\bem\s+parte',                    # In part

#         # Appeal denials
#         r'\bnegado\s+provimento',           # Appeal denied
#         r'\brecurso.*improcedente',         # Appeal unsuccessful
#     ]

#     # Check for STRICTLY favorable first (must be complete victory)
#     for pattern in strictly_favorable_patterns:
#         if re.search(pattern, text_clean):
#             return 'FAVORÁVEL'

#     # Then check for unfavorable patterns
#     for pattern in unfavorable_patterns:
#         if re.search(pattern, text_clean):
#             return 'DESFAVORÁVEL'

#     # Default: if we can't classify clearly, assume unfavorable (conservative approach)
#     return None

In [ ]:
def extract_decision_binary_from_summary_dv(summary_decision):
    """
    DV-only binary classification:
    - DECISÃO ALTERADA
    - DECISÃO MANTIDA

    Logic:
    0) Explicit NEGATED alteration → MANTIDA
    1) Explicit alteration (incl. partials) → ALTERADA
    2) Explicit maintenance → MANTIDA
    3) Otherwise → None
    """
    if not summary_decision or pd.isna(summary_decision):
        return None

    text = str(summary_decision).lower().strip()
    text_clean = re.sub(r"[^\w\s]", " ", text)

    # PRIORITY 0: negated alteration (must come first)
    negated_alteration_patterns = [
        r"\bnegado\s+provimento\b",
        r"\bn[aã]o\s+provido\b",
        r"\bdesprovido\b",
        r"\bnega[rd]\s+provimento\b",
        r"\brecurso\s+improcedente\b",
        r"\bapela[cç][aã]o\s+improcedente\b",
        r"\bimprocedente\b",
        r"\bimprocedência\b",
    ]

    for pattern in negated_alteration_patterns:
        if re.search(pattern, text_clean):
            return "DECISÃO MANTIDA"

    # PRIORITY 1: affirmative alteration (including partials)
    altered_patterns = [
        r"\bparcialmente\s+provido\b",
        r"\bprovido\s+parcialmente\b",
        r"\bprocedente\s+em\s+parte\b",
        r"\bparcial\b",
        r"\bparcialmente\b",
        r"\bem\s+parte\b",
        r"\bprovido\b",
        r"\bprovimento\b",
        r"\bprocedente\b",
        r"\bconcedid[oa]\b",
        r"\balterad[oa]\b",
        r"\balterar\b",
        r"\brevogad[oa]\b",
        r"\brevoga[rd]\b",
        r"\breformad[oa]\b",
    ]

    for pattern in altered_patterns:
        if re.search(pattern, text_clean):
            return "DECISÃO ALTERADA"

    # PRIORITY 2: explicit maintenance
    kept_patterns = [
        r"\bmantid[oa]\b",
        r"\bconfirmad[oa]\b",
        r"\bconfirma[rd]\b",
        r"\bconfirma[çc][aã]o\b",
    ]

    for pattern in kept_patterns:
        if re.search(pattern, text_clean):
            return "DECISÃO MANTIDA"

    return None

In [ ]:
def extract_decision_ternary_from_summary_boc(summary_decision):
    """
    Classificação ternária BoC:
    - FAVORÁVEL
    - DESFAVORÁVEL
    - PARCIAL

    Regras:
    0) Negação explícita tem prioridade absoluta
    1) Parcial é categoria própria
    2) Favorável só se for afirmativo e não negado
    3) Caso ambíguo → None
    """
    if not summary_decision or pd.isna(summary_decision):
        return None

    text = str(summary_decision).lower().strip()
    text_clean = re.sub(r"[^\w\s]", " ", text)

    # PRIORIDADE 0: negação explícita de provimento / procedência
    negated_favorable_patterns = [
        r"\bn[aã]o\s+procede\b",
        r"\bn[aã]o\s+procedente\b",
        r"\bnega[rd]\s+provimento\b",
        r"\bnegado\s+provimento\b",
        r"\bn[aã]o\s+provido\b",
        r"\bdesprovido\b",
        r"\brecurso\s+improcedente\b",
        r"\bapela[cç][aã]o\s+improcedente\b",
        r"\bimprocedente\b",
        r"\bimprocedência\b",
    ]

    for pattern in negated_favorable_patterns:
        if re.search(pattern, text_clean):
            return "DESFAVORÁVEL"

    # PRIORIDADE 1: PARCIAL
    partial_patterns = [
        r"\bparcial\b",
        r"\bparcialmente\b",
        r"\bem\s+parte\b",
        r"\bprocedente\s+em\s+parte\b",
        r"\bprovido\s+parcialmente\b",
        r"\bconcedid[oa]\s+parcialmente\b",
        r"\brevogad[oa]\s+parcialmente\b",
        r"\bconfirmad[oa]\s+em\s+parte\b",
    ]

    for pattern in partial_patterns:
        if re.search(pattern, text_clean):
            return "PARCIAL"

    # PRIORIDADE 2: FAVORÁVEL (afirmativo)
    favorable_patterns = [
        r"\btotalmente\s+procedente\b",
        r"\bprocedente\b",
        r"\bprovido\b",
        r"\bprovimento\b",
        r"\bconcedid[oa]\b",
        r"\brevogad[oa]\b",
        r"\brevog[aou]?\b",
    ]

    for pattern in favorable_patterns:
        if re.search(pattern, text_clean):
            return "FAVORÁVEL"

    # PRIORIDADE 3: DESFAVORÁVEL explícito
    unfavorable_patterns = [
        r"\bmantid[oa]\b",
        r"\bconfirmad[oa]\b",
        r"\bconfirma[çc][aã]o\b",
        r"\bnega[rd]\b",
        r"\bnegad[oa]\b",
    ]

    for pattern in unfavorable_patterns:
        if re.search(pattern, text_clean):
            return "DESFAVORÁVEL"

    return None

In [ ]:
if type_case == "dv":
    df_bin_class = df_class.copy()
    df_bin_class["decisao_binaria"] = df_bin_class["decisao"].apply(
        extract_decision_binary_from_summary_dv
    )
    df_bin_class = df_bin_class.dropna(subset=["decisao_binaria"])

    df_bin_eda = df_eda.copy()
    df_bin_eda["decisao_binaria"] = df_bin_eda["decisao"].apply(
        extract_decision_binary_from_summary_dv
    )
    df_bin_eda = df_bin_eda.dropna(subset=["decisao_binaria"])

    df_final_bin_class = (
        df_bin_class.copy()
    )  # Falta depois o encoding das decisões binárias
    df_final_bin_eda = df_bin_eda.copy()
else:
    df_bin_class = None
    df_bin_eda = None
    df_final_bin_class = None
    df_final_bin_eda = None

In [ ]:
df_final_bin_class

In [ ]:
df_final_bin_eda

In [ ]:
# df_bin_class.to_json(f'./../../data/processed_data/classification/binary/df_acordaos_{type_case}_classification_binary.json', orient='table', force_ascii=False, indent=4)

In [ ]:
# df_final_bin_class.to_csv(Path(f'./../../data/processed_data/classification/binary/df_acordaos_{type_case}_classification_binary.csv'), index=False)
# df_final_bin_eda.to_csv(Path(f'./../../data/processed_data/eda/binary/df_acordaos_{type_case}_eda_binary.csv'), index=False)

In [ ]:
if type_case == "ic":
    df_ter_class = df_class.copy()
    df_ter_class["decisao_ternaria"] = df_ter_class["decisao"].apply(
        extract_decision_ternary_from_summary_boc
    )
    df_ter_class = df_ter_class.dropna(subset=["decisao_ternaria"])

    df_ter_eda = df_eda.copy()
    df_ter_eda["decisao_ternaria"] = df_ter_eda["decisao"].apply(
        extract_decision_ternary_from_summary_boc
    )
    df_ter_eda = df_ter_eda.dropna(subset=["decisao_ternaria"])

    df_final_ter_class = (
        df_ter_class.copy()
    )  # Falta depois o encoding das decisões binárias
    df_final_ter_eda = df_ter_eda.copy()
else:
    df_ter_class = None
    df_ter_eda = None
    df_final_ter_class = None
    df_final_ter_eda = None

In [ ]:
df_final_ter_class

In [ ]:
# df_final_ter_class.to_json(f'./../../data/processed_data/classification/ternary/df_acordaos_{type_case}_classification_ternary.json', orient='table', force_ascii=False, indent=4)

In [ ]:
df_final_ter_eda

In [ ]:
# df_final_ter_class.to_csv(Path(f'./../../data/processed_data/classification/ternary/df_acordaos_{type_case}_classification_ternary.csv'), index=False)
# df_final_ter_eda.to_csv(Path(f'./../../data/processed_data/eda/ternary/df_acordaos_{type_case}_eda_ternary.csv'), index=False)

In [ ]:
# Count DESFAVORÁVEL in binary classification
desfavoravel_bin = (df_final_bin_class["decisao_binaria"] == "DESFAVORÁVEL").sum()

# Count TOTALMENTE DESFAVORÁVEL in ternary classification
desfavoravel_ter = (
    df_final_ter_class["decisao_ternaria"] == "TOTALMENTE DESFAVORÁVEL"
).sum()

# Count PARCIAL in ternary classification
parcial_ter = (df_final_ter_class["decisao_ternaria"] == "PARCIAL").sum()

print(f"DESFAVORÁVEL (binary): {desfavoravel_bin}")
print(f"TOTALMENTE DESFAVORÁVEL (ternary): {desfavoravel_ter}")
print(f"PARCIAL (ternary): {parcial_ter}")
print(f"Sum of TOTALMENTE DESFAVORÁVEL + PARCIAL: {desfavoravel_ter + parcial_ter}")
print(f"\nDifference: {desfavoravel_bin - (desfavoravel_ter + parcial_ter)}")

## Preparação Classificação

USANDO df_final_bin_class e df_final_ter_class


- removal of cases prior to 2007 for ic
- dividing timeseries_split. then do class imbalance mitigation on training split. leave test alone

- stopwords removal(ML only). If possible legal stop words removal (not mandatory, wont do it)
- lematization(ML only)
- bert tokenization on dl (not nltk)
- law articles detection (problably only in DL/LLMs tho), removal from main text  and storage outside the training data but correcly indexed to its specific case.
-  feature engineering (baseline can be tf-idf)
- 

In [9]:
import pandas as pd

type_case = "dv"  # 'dv' para Violência Doméstica, 'ic' para Incumprimento de Contratos
type_case_verbose = (
    "Violência Doméstica" if type_case == "dv" else "Incumprimento de Contratos"
)
type_case_folder = "contract_breach" if type_case == "boc" else "domestic_violence"
type_analysis = "classification"  # 'classification' ou 'eda'
df_bin_class = pd.read_csv(
    f"./../../data/processed_data/{type_analysis}/binary/df_acordaos_{type_case}_{type_analysis}_binary.csv"
)
# df_ter_class = pd.read_csv(
#     f"./../../data/processed_data/{type_analysis}/ternary/df_acordaos_{type_case}_{type_analysis}_ternary.csv"
# )

In [10]:
len(df_bin_class)

1126

## EDA (scripted)

EDA is now in a lean script. Run the cell below for either `dv` or `boc`.

In [ ]:
from pathlib import Path
import sys

base_dir = Path("./../../").resolve()
sys.path.insert(0, str(base_dir / "src"))

from eda.eda import run_case_eda

# Choose: 'dv' or 'boc'
run_case_eda(case_type="boc", base_dir=base_dir)

In [ ]:
import pandas as pd
from pathlib import Path
import sys

base_dir = Path("./../../").resolve()
sys.path.insert(0, str(base_dir / "src"))

df_dv_origin = pd.read_csv(
    f"./../../data/processed_data/eda/binary/df_acordaos_dv_eda_binary.csv"
)
df_boc_origin = pd.read_csv(
    f"./../../data/processed_data/eda/ternary/df_acordaos_boc_eda_ternary.csv"
)

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("./../../").resolve()

# === Set your cutoffs here ===
DV_CUTOFF = "2025-05-01"
BOC_CUTOFF = "2023-10-01"


def split_by_user_cutoff(
    df: pd.DataFrame,
    date_col: str,
    cutoff_date: str,
    n: int = 50,
    include_missing_in_train: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()
    df["_acordao_date"] = pd.to_datetime(df[date_col], dayfirst=True, errors="coerce")

    cutoff = pd.to_datetime(cutoff_date)
    df_valid = df[df["_acordao_date"].notna()]
    df_missing = df[df["_acordao_date"].isna()]

    test_pool = df_valid[df_valid["_acordao_date"] >= cutoff].sort_values(
        "_acordao_date", ascending=False
    )

    if len(test_pool) < n:
        raise ValueError(
            f"Not enough cases on/after cutoff {cutoff_date}. "
            f"Needed {n}, found {len(test_pool)}."
        )

    test_df = test_pool.head(n)

    train_df = df_valid[df_valid["_acordao_date"] < cutoff]
    if include_missing_in_train:
        train_df = pd.concat([train_df, df_missing], ignore_index=False)

    return test_df, train_df


dv_df = pd.read_csv(
    BASE_DIR / "data/processed_data/eda/binary/df_acordaos_dv_eda_binary.csv"
)
boc_df = pd.read_csv(
    BASE_DIR / "data/processed_data/eda/ternary/df_acordaos_boc_eda_ternary.csv"
)

dv_test, dv_train = split_by_user_cutoff(
    dv_df, "data_acordao", DV_CUTOFF, n=50, include_missing_in_train=True
)
boc_test, boc_train = split_by_user_cutoff(
    boc_df, "data_acordao", BOC_CUTOFF, n=50, include_missing_in_train=True
)

print("DV test size:", len(dv_test), "DV train size:", len(dv_train))
print("BOC test size:", len(boc_test), "BOC train size:", len(boc_train))

# Optional save
out_dir = BASE_DIR / "data/processed_data" / "splits" / "gold_test"
out_dir.mkdir(parents=True, exist_ok=True)

dv_test.drop(columns=["_acordao_date"]).to_csv(out_dir / "dv_gold_test.csv", index=False)
boc_test.drop(columns=["_acordao_date"]).to_csv(out_dir / "boc_gold_test.csv", index=False)

dv_train.drop(columns=["_acordao_date"]).to_csv(out_dir / "dv_train_before_cutoff.csv", index=False)
boc_train.drop(columns=["_acordao_date"]).to_csv(out_dir / "boc_train_before_cutoff.csv", index=False)

In [ ]:
boc_train

In [ ]:
boc_test

In [ ]:
dv_test

In [ ]:
# Attach full text to gold test sets using url
def attach_full_text(
    test_df, df_all, name, url_col="url", text_col="texto_integral_completo"
):
    # Guardrails: required columns
    for col in [url_col, text_col]:
        if col not in df_all.columns:
            raise KeyError(f"{name}: column '{col}' not found in df_all")
        if col not in test_df.columns and col != text_col:
            raise KeyError(f"{name}: column '{col}' not found in test_df")

    # Enforce unique urls in source to avoid row explosion
    dup_count = df_all[url_col].duplicated().sum()
    if dup_count:
        raise ValueError(
            f"{name}: {dup_count} duplicate urls in df_all; cannot safely merge"
        )

    merged = test_df.merge(
        df_all[[url_col, text_col]],
        on=url_col,
        how="left",
        validate="one_to_one",
    )

    missing = merged[text_col].isna().sum()
    if missing:
        examples = merged.loc[merged[text_col].isna(), url_col].head(5).tolist()
        raise ValueError(
            f"{name}: {missing} cases missing full text. Example URLs: {examples}"
        )

    return merged


dv_test_full = attach_full_text(dv_test, df_dv_all, name="DV")
boc_test_full = attach_full_text(
    boc_test, df_ic_all, name="BOC"
)  # BOC maps to IC in df name

dv_test_full.head()
boc_test_full.head()

In [ ]:
# Drop columns: 'tribunal', 'juiz_relator' and 5 other columns
boc_test_full = boc_test_full.drop(
    columns=[
        "tribunal",
        "juiz_relator",
        "data_acordao",
        "descritores",
        "decisao",
        "_acordao_date",
    ]
)
# Drop columns: 'tribunal', 'juiz_relator' and 5 other columns
dv_test_full = dv_test_full.drop(
    columns=[
        "tribunal",
        "juiz_relator",
        "data_acordao",
        "descritores",
        "decisao",
        "_acordao_date",
    ]
)

In [ ]:
dv_test_full

In [ ]:
boc_test_full

In [ ]:
out_dir_test = BASE_DIR / "data/processed_data" / "gold_test"
out_dir.mkdir(parents=True, exist_ok=True)

dv_test_full.to_csv(out_dir / "dv_gold_test_full.csv", index=False)
boc_test_full.to_csv(out_dir / "boc_gold_test_full.csv", index=False)

dv_test_full.to_json(
    out_dir / "dv_gold_test_full.json", orient="table", force_ascii=False, indent=4
)
boc_test_full.to_json(
    out_dir / "boc_gold_test_full.json", orient="table", force_ascii=False, indent=4
)

## Embeddigns and UMAP


In [ ]:
import pandas as pd
from pathlib import Path

dv_embedding_path = Path(
    "./../../data/processed_data/bert_tokens_embedd/df_acordaos_dv_eda_binary_legal_bert_embeddings.parquet"
)

boc_embedding_path = Path(
    "./../../data/processed_data/bert_tokens_embedd/df_acordaos_boc_eda_ternary_legal_bert_embeddings.parquet"
)

df_dv_embeddings = pd.read_parquet(dv_embedding_path, engine="fastparquet")
df_boc_embeddings = pd.read_parquet(boc_embedding_path, engine="fastparquet")

display(df_dv_embeddings.head())
display(df_boc_embeddings.head())

In [ ]:
from pathlib import Path
import sys

base_dir = Path("./../../").resolve()
sys.path.insert(0, str(base_dir / "src"))
from eda.eda import plotly_umap_3d

df_dv_umap, fig_dv = plotly_umap_3d(
    parquet_path="df_acordaos_dv_eda_binary_legal_bert_embeddings.parquet",
    base_dir=base_dir,
    sample_size=3000,
    n_neighbors=30,
    min_dist=0.5,
    save_html=True,
    show_figure=True,
    title="UMAP 3D - Domestic Violence (Non-tuned BERT Embeddings)",
)


df_boc_umap, fig_boc = plotly_umap_3d(
    parquet_path="df_acordaos_boc_eda_ternary_legal_bert_embeddings.parquet",
    base_dir=base_dir,
    sample_size=3000,
    n_neighbors=30,
    min_dist=0.5,
    save_html=True,
    show_figure=True,
    title="UMAP 3D - Breach of Contract (Non-tuned BERT Embeddings)",
)

In [ ]:
from pathlib import Path
import sys

base_dir = Path("./../../").resolve()
sys.path.insert(0, str(base_dir / "src"))
from eda.eda import plotly_umap_3d

df_dv_umap, fig_dv = plotly_umap_3d(
    parquet_path="bert_v4_dv_finetuned_embeddings.parquet",
    base_dir=base_dir,
    sample_size=3000,
    n_neighbors=30,
    min_dist=0.5,
    save_html=True,
    show_figure=True,
    title="UMAP3D - Domestic Violence (Finetuned BERT Embeddings)",
)


df_boc_umap, fig_boc = plotly_umap_3d(
    parquet_path="bert_v4_boc_finetuned_embeddings.parquet",
    base_dir=base_dir,
    sample_size=3000,
    n_neighbors=30,
    min_dist=0.5,
    save_html=True,
    show_figure=True,
    title="UMAP3D - Breach of Contract (Finetuned BERT Embeddings)",
)

In [ ]:
from pathlib import Path
import sys

base_dir = Path("./../../").resolve()
sys.path.insert(0, str(base_dir / "src"))

from eda.eda import plotly_umap_3d_combined

# Combine both case types in one UMAP visualization
df_combined, fig_combined = plotly_umap_3d_combined(
    dv_parquet_path="df_acordaos_dv_eda_binary_legal_bert_embeddings.parquet",
    boc_parquet_path="df_acordaos_boc_eda_ternary_legal_bert_embeddings.parquet",
    base_dir=base_dir,
    sample_size=3000,
    n_neighbors=30,
    min_dist=0.5,
    save_html=True,
    show_figure=True,
    title="UMAP 3D - Combined DV & BOC Embeddings",
)


# TODO:
 

- [] CLASSIFICAÇÃO - AFINAR PLANO 
- [] Escrever fundamentos teoricos para tese
- [] Escrever relatorio para IICD
- []  Organizar df's para classificação (splits equilibrados, processamento necessario, data augmentation, featurizers necessarios e modelos treino e avaliação necessária (se possivel com mlflow))




